In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist
from sklearn.cluster import KMeans
import pickle
def makefile(what,filename):
    with open(filename,"wb") as f3:
        pickle.dump(what,f3)

def readfile(filename):
    with open(filename,"rb") as f4:
        ans=pickle.load(f4)
    return ans
from math import *

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

2026-02-04 13:03:52.881860: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/toshihayashi/anaconda3/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import warnings
warnings.filterwarnings("ignore")

In [7]:
#download Kuzushiji49.

X_train=np.load("k49-train-imgs.npz")["arr_0"]
y_train=np.load("k49-train-labels.npz")["arr_0"]
X_test=np.load("k49-test-imgs.npz")["arr_0"]
y_test=np.load("k49-test-labels.npz")["arr_0"]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(49):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)  

2
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
[77.6, 62.1, 89.3, 61.4, 72.3, 65.6, 62.3, 66.1, 66.4, 61.6, 73.2, 63.8, 64.6, 69.1, 63.1, 66.0, 75.7, 55.4, 66.2, 60.0, 70.2, 60.4, 75.6, 76.3, 64.4, 74.7, 76.9, 56.9, 79.4, 61.8, 56.4, 57.5, 72.1, 75.0, 71.5, 63.3, 72.3, 78.7, 81.9, 75.8, 58.7, 68.4, 70.7, 67.3, 72.5, 84.6, 64.4, 81.6, 63.1]
[74.9, 59.7, 85.9, 62.4, 67.7, 59.9, 56.6, 60.6, 65.0, 57.5, 65.0, 55.1, 60.1, 69.3, 57.4, 59.7, 68.8, 59.3, 60.8, 51.9, 69.2, 57.0, 75.2, 77.2, 62.3, 68.7, 73.5, 55.0, 74.1, 62.7, 52.0, 57.7, 70.0, 71.1, 65.2, 62.2, 71.7, 72.1, 75.8, 71.4, 55.0, 67.5, 66.3, 61.9, 69.8, 80.3, 63.5, 75.0, 56.8]
[80.9, 77.7, 91.9, 82.3, 72.7, 75.0, 80.4, 83.6, 74.9, 70.6, 83.9, 73.7, 71.6, 69.6, 80.3, 78.3, 85.9, 85.8, 83.3, 82.3, 73.2, 66.4, 79.6, 76.7, 81.6, 78.3, 83.2, 73.9, 77.3, 72.3, 71.8, 74.0, 79.7, 83.8, 81.5, 85.1, 82.8, 83.5, 79.4, 75.2, 74.5, 68.2, 87.1, 73.9, 76.6, 

43
44
45
46
47
48
[74.8, 71.2, 92.1, 63.7, 62.5, 71.9, 66.2, 67.3, 51.8, 69.7, 80.7, 61.0, 65.2, 81.7, 74.2, 65.4, 93.0, 51.1, 81.6, 68.6, 66.4, 68.4, 87.2, 69.5, 63.6, 62.0, 81.1, 70.8, 81.1, 56.7, 71.0, 55.3, 78.4, 81.0, 68.8, 65.0, 72.2, 86.5, 80.7, 69.5, 68.2, 60.7, 76.7, 65.0, 82.6, 91.2, 73.9, 85.8, 57.0]
[79.8, 79.8, 92.3, 71.0, 70.3, 82.0, 76.9, 83.3, 62.1, 75.8, 81.3, 76.7, 73.8, 81.9, 77.2, 69.7, 91.5, 67.9, 88.3, 75.6, 76.9, 74.3, 88.5, 75.9, 77.2, 79.2, 81.2, 70.6, 84.7, 63.1, 77.4, 68.8, 78.9, 85.9, 77.5, 67.7, 82.5, 91.1, 84.6, 79.7, 74.4, 76.6, 83.4, 70.7, 85.0, 90.6, 81.3, 89.5, 67.1]
[79.8, 91.8, 96.8, 94.1, 84.4, 84.7, 90.3, 92.5, 80.9, 86.9, 85.5, 84.6, 82.1, 69.1, 78.4, 89.0, 94.3, 88.9, 92.9, 86.3, 71.0, 86.9, 83.5, 80.4, 80.4, 88.8, 90.3, 86.1, 89.2, 85.0, 81.8, 89.0, 93.3, 88.3, 84.9, 78.6, 84.3, 92.2, 92.3, 88.8, 77.5, 80.5, 92.0, 90.6, 84.2, 83.6, 84.6, 95.1, 85.3]
[66.6, 84.3, 96.5, 94.8, 78.2, 76.1, 79.3, 90.8, 65.0, 71.6, 86.5, 75.8, 78.1, 85.9, 81.2, 75.9, 

In [8]:
X_train=np.load("k49-train-imgs.npz")["arr_0"]
y_train=np.load("k49-train-labels.npz")["arr_0"]
X_test=np.load("k49-test-imgs.npz")["arr_0"]
y_test=np.load("k49-test-labels.npz")["arr_0"]
num_cluster=[2,3,5,8,10,16,25]
window_size=[7]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(49):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)  

7
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
[62.7, 53.0, 80.6, 62.4, 58.4, 59.6, 52.9, 60.8, 58.6, 54.7, 60.5, 61.9, 62.1, 65.7, 56.1, 58.5, 60.7, 55.5, 60.4, 53.3, 65.0, 52.7, 63.0, 61.5, 49.0, 66.5, 58.1, 47.7, 70.2, 50.9, 52.9, 55.7, 57.1, 67.1, 57.1, 60.9, 57.9, 64.4, 72.5, 64.3, 60.3, 60.4, 59.3, 60.6, 63.3, 66.8, 55.5, 67.4, 51.4]
[62.7, 52.6, 80.4, 61.4, 58.0, 59.4, 53.4, 61.3, 57.3, 53.7, 59.7, 66.5, 62.1, 65.8, 56.5, 58.3, 60.2, 55.1, 61.1, 56.1, 63.0, 53.4, 62.3, 61.1, 49.8, 66.7, 59.3, 47.4, 71.0, 53.0, 51.7, 55.7, 57.0, 66.9, 57.3, 62.4, 60.1, 64.4, 73.1, 64.3, 59.5, 59.8, 60.4, 60.5, 64.3, 67.9, 55.6, 67.7, 51.3]
[55.3, 52.0, 68.8, 62.6, 54.3, 52.3, 50.4, 52.7, 50.2, 50.4, 54.3, 56.6, 53.3, 53.9, 51.1, 53.3, 53.6, 50.9, 55.5, 51.3, 54.4, 50.7, 55.0, 51.6, 49.3, 55.8, 51.5, 48.2, 57.2, 49.9, 49.6, 54.4, 51.0, 54.9, 51.0, 53.8, 54.0, 55.5, 58.2, 57.4, 52.2, 54.1, 53.2, 55.2, 59.3, 

47
48
[80.2, 56.4, 91.1, 58.4, 65.7, 68.9, 62.6, 64.8, 65.2, 64.5, 73.4, 62.8, 68.0, 71.6, 70.6, 66.9, 82.0, 44.0, 76.4, 66.3, 67.5, 65.0, 79.2, 76.2, 68.4, 76.3, 78.4, 65.3, 73.8, 56.9, 63.1, 56.3, 70.5, 79.7, 71.5, 69.8, 66.4, 82.6, 80.4, 77.9, 61.6, 66.8, 73.3, 68.6, 77.9, 86.0, 66.2, 86.5, 68.7]
[81.5, 60.8, 88.8, 64.4, 65.7, 67.9, 59.0, 64.8, 69.8, 63.0, 75.6, 62.2, 65.7, 73.6, 69.4, 64.6, 79.9, 55.6, 75.1, 65.2, 72.6, 64.6, 82.6, 79.5, 69.6, 74.9, 81.5, 63.0, 73.1, 62.8, 62.8, 57.8, 71.7, 81.1, 69.5, 72.0, 73.1, 79.3, 76.4, 74.5, 62.3, 66.8, 74.3, 66.6, 78.6, 87.9, 69.6, 84.9, 69.4]
[79.3, 86.7, 96.1, 90.5, 79.6, 80.4, 84.5, 88.9, 75.7, 81.0, 83.9, 76.9, 81.3, 69.6, 82.5, 80.9, 90.9, 86.6, 89.5, 90.9, 71.7, 78.1, 83.2, 68.0, 82.5, 79.6, 76.6, 81.9, 82.9, 81.3, 82.8, 82.2, 86.1, 79.8, 84.4, 83.7, 78.9, 90.5, 90.9, 86.7, 80.1, 77.7, 89.9, 78.2, 79.6, 90.3, 84.2, 92.1, 93.2]
[80.9, 72.4, 94.3, 76.8, 69.0, 74.4, 76.0, 80.2, 68.4, 69.8, 77.7, 83.3, 71.8, 71.3, 75.4, 73.5, 88.3, 67.8, 

In [9]:
X_train=np.load("k49-train-imgs.npz")["arr_0"]
y_train=np.load("k49-train-labels.npz")["arr_0"]
X_test=np.load("k49-test-imgs.npz")["arr_0"]
y_test=np.load("k49-test-labels.npz")["arr_0"]
num_cluster=[2,3,5,8,10,16,25]
window_size=[15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(49):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)  

15
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
[67.6, 66.3, 82.7, 60.6, 56.4, 52.3, 59.4, 74.2, 54.5, 59.8, 76.1, 65.9, 50.3, 51.8, 59.4, 64.3, 61.2, 60.2, 60.3, 62.2, 48.3, 59.3, 66.8, 64.5, 60.6, 60.8, 62.2, 53.7, 63.9, 55.0, 64.9, 51.7, 64.7, 65.3, 64.3, 46.7, 55.1, 72.6, 57.5, 56.2, 56.3, 60.5, 59.5, 62.8, 64.0, 59.9, 64.4, 78.6, 49.0]
[67.0, 66.6, 83.0, 56.5, 56.0, 52.7, 60.8, 73.7, 55.1, 60.3, 75.8, 62.9, 55.4, 52.3, 59.7, 62.4, 60.5, 60.0, 57.8, 63.6, 51.1, 58.1, 67.7, 64.8, 59.9, 61.4, 62.8, 53.2, 64.0, 53.3, 64.9, 54.2, 65.4, 64.1, 63.6, 47.1, 60.0, 72.3, 59.7, 56.9, 55.5, 61.9, 58.9, 60.2, 63.7, 60.0, 63.1, 78.7, 53.0]
[56.5, 55.7, 70.8, 56.8, 50.1, 50.5, 52.2, 55.7, 50.4, 52.2, 60.4, 53.7, 50.4, 50.5, 54.6, 53.5, 54.9, 52.4, 51.5, 56.3, 51.3, 51.7, 59.7, 58.7, 53.7, 53.1, 55.6, 51.2, 53.8, 54.1, 51.2, 51.6, 60.3, 59.2, 52.4, 47.4, 49.1, 59.3, 51.0, 51.0, 50.8, 54.9, 54.7, 55.4, 60.9,

47
48
[74.6, 65.6, 91.7, 71.6, 63.4, 73.2, 62.9, 67.3, 51.2, 71.0, 81.5, 64.8, 67.7, 80.4, 72.4, 63.9, 88.9, 53.9, 77.0, 71.0, 65.8, 67.8, 85.3, 62.3, 65.5, 65.8, 74.4, 69.7, 78.2, 60.1, 68.9, 53.7, 76.0, 80.9, 70.0, 72.5, 66.4, 85.9, 79.3, 72.3, 66.8, 51.0, 83.9, 65.7, 80.7, 88.0, 77.3, 86.3, 64.2]
[78.9, 75.5, 91.6, 74.9, 74.1, 76.7, 69.8, 81.1, 56.9, 74.2, 82.7, 74.5, 73.3, 80.0, 74.1, 67.5, 88.6, 66.5, 87.1, 76.3, 73.3, 70.7, 87.0, 67.5, 72.3, 75.1, 76.4, 68.9, 83.3, 62.2, 73.8, 66.4, 75.6, 83.9, 76.1, 74.8, 77.5, 88.5, 80.7, 76.0, 72.7, 75.8, 84.0, 68.8, 82.7, 88.4, 79.6, 89.7, 70.4]
[74.2, 88.4, 96.1, 91.3, 81.1, 75.5, 83.1, 90.9, 73.9, 82.1, 81.4, 79.5, 76.8, 55.0, 72.3, 85.5, 92.2, 78.9, 91.4, 90.6, 69.3, 74.8, 82.7, 75.9, 80.0, 84.5, 87.0, 85.3, 79.4, 77.9, 76.1, 82.7, 88.5, 76.8, 78.0, 65.8, 77.5, 89.1, 88.0, 84.2, 74.3, 77.0, 76.2, 84.5, 80.1, 77.0, 81.4, 94.4, 82.0]
[81.0, 83.9, 93.9, 88.6, 76.5, 79.7, 76.0, 85.7, 60.7, 79.5, 85.4, 82.0, 79.6, 83.8, 79.7, 79.0, 93.6, 82.5, 